<a href="https://colab.research.google.com/github/sk27110/DL2/blob/HW-6-8/%D0%94%D0%976_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Протестируем модели с классическим вниманием и вниманием из статьи на корпусе Flowers 102. Это корпус для fine-grained многоклассовой классификации. Посмотрим на обучение моделей при примерно равном числе параметров, замерим accuracy, время на эпоху и использование памяти. Реализацию MHLA возьмем свою, так как оригинальный код не запускается из-за проблем с библиотеками.

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [2]:
import os
import time
import math
import random
import copy
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from torchvision.datasets import Flowers102

from torch.cuda.amp import autocast, GradScaler


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [4]:
@dataclass
class Config:
    data_root: str = "./data"

    image_size: int = 128
    patch_size: int = 4
    num_classes: int = 102

    batch_size: int = 64
    num_workers: int = 0

    epochs: int = 60
    lr: float = 4e-4
    weight_decay: float = 0.03

    embed_dim: int = 128
    depth: int = 3
    num_heads: int = 4
    mlp_ratio: float = 4.0
    dropout: float = 0.1

    use_amp: bool = False


cfg = Config()

print(cfg)
print("Number of image tokens:", (cfg.image_size // cfg.patch_size) ** 2)


Config(data_root='./data', image_size=128, patch_size=4, num_classes=102, batch_size=64, num_workers=0, epochs=60, lr=0.0004, weight_decay=0.03, embed_dim=128, depth=3, num_heads=4, mlp_ratio=4.0, dropout=0.1, use_amp=False)
Number of image tokens: 1024


In [5]:
train_transform = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

eval_transform = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

train_dataset = Flowers102(
    root=cfg.data_root,
    split="train",
    download=True,
    transform=train_transform,
)

val_dataset = Flowers102(
    root=cfg.data_root,
    split="val",
    download=True,
    transform=eval_transform,
)

test_dataset = Flowers102(
    root=cfg.data_root,
    split="test",
    download=True,
    transform=eval_transform,
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


100%|██████████| 345M/345M [00:12<00:00, 26.5MB/s]
100%|██████████| 502/502 [00:00<00:00, 877kB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 43.7MB/s]


Train: 1020
Val: 1020
Test: 6149


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
)


In [16]:
class PatchEmbed(nn.Module):
    def __init__(self, image_size=128, patch_size=8, in_chans=3, embed_dim=192):
        super().__init__()

        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"

        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size * self.grid_size

        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


In [17]:
class StandardAttention(nn.Module):
    def __init__(self, dim, num_heads=3, attn_drop=0.0, proj_drop=0.0):
        super().__init__()

        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).reshape(B, N, C)

        out = self.proj(out)
        out = self.proj_drop(out)

        return out


In [18]:
class SequenceLabZeroSAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        dropout=0.0,
        block_size=1025,
        bias=True,
    ):
        super().__init__()

        from sequencelab.build import build_attention
        from sequencelab.config import ZeroSConfig

        cfg = ZeroSConfig(
            n_embd=dim,
            n_head=num_heads,
            dropout=dropout,
            bias=bias,
            block_size=block_size,
            is_causal=False,
            init_params=False,
            init_n_layers=1,
            use_norm=True,
            use_associative=True,
        )

        self.attn = build_attention(cfg)

    def forward(self, x):
        return self.attn(x)


In [22]:
class SequenceLabFEMAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        dropout=0.0,
        prior_type="softmax",
        bias=True,
    ):
        super().__init__()

        from sequencelab.build import build_attention
        from sequencelab.config import FEMConfig

        cfg = FEMConfig(
            n_embd=dim,
            n_head=num_heads,
            prior_type=prior_type,
            dropout=dropout,
            causal=False,
            bias=bias,

            fem_ratio=0.5,
            p_t_to_fem_ratio=2.0,

            use_temperature=True,
            use_lse=True,
            use_outer_gate=True,

            use_rope=False,
            use_conv=False,
            conv_hidden=64,
            conv_norm_first=True,
            conv_bidirectional=False,
        )

        self.attn = build_attention(cfg)

    def forward(self, x):
        return self.attn(x)


In [38]:
class ELSAAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        attn_drop=0.0,
        proj_drop=0.0,
        backend="auto",
    ):
        super().__init__()

        from stable.elsa import ElsaAttention

        self.attn = ElsaAttention(
            dim=dim,
            num_heads=num_heads,
            attn_drop=attn_drop,
            proj_drop=proj_drop,
            backend=backend,
        )

    def forward(self, x):
        return self.attn(x)


In [19]:
class MHLALinearAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=3,
        attn_drop=0.0,
        proj_drop=0.0,
        eps=1e-6,
    ):
        super().__init__()

        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.eps = eps

        self.qkv = nn.Linear(dim, dim * 3, bias=True)

        self.head_gate = nn.Linear(dim, num_heads, bias=True)

        self.local_conv = nn.Conv2d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
            bias=True,
        )

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def feature_map(self, x):
        return F.elu(x) + 1.0

    def local_positional_encoding(self, x):
        B, N, C = x.shape

        cls_token = x[:, :1, :]
        patch_tokens = x[:, 1:, :]

        num_patches = patch_tokens.shape[1]
        grid_size = int(math.sqrt(num_patches))

        if grid_size * grid_size != num_patches:
            return torch.zeros_like(x)

        feat = patch_tokens.transpose(1, 2).reshape(B, C, grid_size, grid_size)
        feat = self.local_conv(feat)
        feat = feat.flatten(2).transpose(1, 2)

        zero_cls = torch.zeros_like(cls_token)
        return torch.cat([zero_cls, feat], dim=1)

    def forward(self, x):
        B, N, C = x.shape

        local = self.local_positional_encoding(x)

        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]  # [B, H, N, D]

        q = self.feature_map(q)
        k = self.feature_map(k)

        v = self.attn_drop(v)

        kv = torch.einsum("bhnd,bhne->bhde", k, v)

        k_sum = k.sum(dim=2)
        z = 1.0 / (torch.einsum("bhnd,bhd->bhn", q, k_sum) + self.eps)

        out = torch.einsum("bhnd,bhde,bhn->bhne", q, kv, z)

        gates = torch.softmax(self.head_gate(x), dim=-1)

        out = out.permute(0, 2, 1, 3)
        out = out * gates.unsqueeze(-1) * self.num_heads
        out = out.reshape(B, N, C)

        out = out + local

        out = self.proj(out)
        out = self.proj_drop(out)

        return out


In [40]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        dropout=0.1,
        attention_type="standard",
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        if attention_type == "standard":
            self.attn = StandardAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
            )

        elif attention_type == "mhla":
            self.attn = MHLALinearAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
            )

        elif attention_type == "sequencelab_zeros":
            self.attn = SequenceLabZeroSAttention(
                dim=dim,
                num_heads=num_heads,
                dropout=dropout,
                block_size=2048,
                bias=True,
            )

        elif attention_type == "sequencelab_fem_softmax":
            self.attn = SequenceLabFEMAttention(
                dim=dim,
                num_heads=num_heads,
                dropout=dropout,
                prior_type="softmax",
                bias=True,
            )

        elif attention_type == "sequencelab_fem_linear":
            self.attn = SequenceLabFEMAttention(
                dim=dim,
                num_heads=num_heads,
                dropout=dropout,
                prior_type="linear",
                bias=True,
            )
        elif attention_type == "elsa_auto":
            self.attn = ELSAAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
                backend="auto",
            )

        elif attention_type == "elsa_triton":
            self.attn = ELSAAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
                backend="triton",
            )

        elif attention_type == "elsa_pytorch":
            self.attn = ELSAAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
                backend="pytorch",
            )

        elif attention_type == "elsa_triton_fp32_train":
            self.attn = ELSAAttention(
                dim=dim,
                num_heads=num_heads,
                attn_drop=dropout,
                proj_drop=dropout,
                backend="triton_fp32_train",
            )


        else:
            raise ValueError(f"Unknown attention_type: {attention_type}")

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(
            dim=dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


In [24]:
class TinyViT(nn.Module):
    def __init__(
        self,
        image_size=128,
        patch_size=8,
        in_chans=3,
        num_classes=102,
        embed_dim=192,
        depth=4,
        num_heads=3,
        mlp_ratio=4.0,
        dropout=0.1,
        attention_type="standard",
    ):
        super().__init__()

        self.patch_embed = PatchEmbed(
            image_size=image_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
        )

        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                attention_type=attention_type,
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out")
            if m.bias is not None:
                nn.init.zeros_(m.bias)

        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B = x.size(0)

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        cls = x[:, 0]
        logits = self.head(cls)

        return logits


In [43]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

standard_model = TinyViT(
    image_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=cfg.num_classes,
    embed_dim=cfg.embed_dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
    attention_type="standard",
)

mhla_model = TinyViT(
    image_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=cfg.num_classes,
    embed_dim=cfg.embed_dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
    attention_type="mhla",
)

seq_model = TinyViT(
    image_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=cfg.num_classes,
    embed_dim=cfg.embed_dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
    attention_type="sequencelab_fem_softmax",
)

elsa_model = TinyViT(
    image_size=cfg.image_size,
    patch_size=cfg.patch_size,
    num_classes=cfg.num_classes,
    embed_dim=cfg.embed_dim,
    depth=cfg.depth,
    num_heads=cfg.num_heads,
    mlp_ratio=cfg.mlp_ratio,
    dropout=cfg.dropout,
    attention_type="elsa_auto",
)

print("ELSA params:", count_params(elsa_model))



print("Standard params:", count_params(standard_model))
print("MHLA params:", count_params(mhla_model))
print("SequenceLab FEM params:", count_params(seq_model))


ELSA params: 744678
Standard params: 745830
MHLA params: 751218
SequenceLab FEM params: 696678


In [29]:
def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)
    correct = (preds == targets).sum().item()
    total = targets.size(0)
    return correct, total


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    use_amp=True,
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    pbar = tqdm(loader, desc="train", leave=False)

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(use_amp and device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = targets.size(0)

        running_loss += loss.item() * batch_size
        correct, total = accuracy_top1(logits.detach(), targets)
        running_correct += correct
        running_total += total

        pbar.set_postfix({
            "loss": running_loss / running_total,
            "acc": running_correct / running_total,
        })

    epoch_time = time.time() - start_time

    return {
        "loss": running_loss / running_total,
        "acc": running_correct / running_total,
        "time": epoch_time,
    }


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    use_amp=True,
    desc="eval",
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time()

    pbar = tqdm(loader, desc=desc, leave=False)

    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(enabled=(use_amp and device.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, targets)

        batch_size = targets.size(0)

        running_loss += loss.item() * batch_size
        correct, total = accuracy_top1(logits, targets)
        running_correct += correct
        running_total += total

        pbar.set_postfix({
            "loss": running_loss / running_total,
            "acc": running_correct / running_total,
        })

    eval_time = time.time() - start_time

    return {
        "loss": running_loss / running_total,
        "acc": running_correct / running_total,
        "time": eval_time,
    }


In [42]:
def run_experiment(attention_type, cfg):
    allowed_attention_types = [
        "standard",
        "mhla",
        "sequencelab_zeros",
        "sequencelab_fem_softmax",
        "sequencelab_fem_linear",
        "elsa_auto",
        "elsa_triton",
        "elsa_pytorch",
        "elsa_triton_fp32_train",
    ]



    assert attention_type in allowed_attention_types


    set_seed(42)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = TinyViT(
        image_size=cfg.image_size,
        patch_size=cfg.patch_size,
        num_classes=cfg.num_classes,
        embed_dim=cfg.embed_dim,
        depth=cfg.depth,
        num_heads=cfg.num_heads,
        mlp_ratio=cfg.mlp_ratio,
        dropout=cfg.dropout,
        attention_type=attention_type,
    ).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg.epochs,
    )

    scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))

    history = []

    best_val_acc = -1.0
    best_state = None

    print(f"\n===== Experiment: {attention_type} =====")
    print("Params:", count_params(model))

    total_start = time.time()

    for epoch in range(1, cfg.epochs + 1):
        print(f"\nEpoch {epoch}/{cfg.epochs}")

        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            use_amp=cfg.use_amp,
        )

        val_metrics = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            use_amp=cfg.use_amp,
            desc="val",
        )

        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]

        row = {
            "attention": attention_type,
            "epoch": epoch,
            "lr": lr_now,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "train_time": train_metrics["time"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "val_time": val_metrics["time"],
        }

        history.append(row)

        print(
            f"train_loss={row['train_loss']:.4f} "
            f"train_acc={row['train_acc']:.4f} | "
            f"val_loss={row['val_loss']:.4f} "
            f"val_acc={row['val_acc']:.4f} | "
            f"lr={lr_now:.2e}"
        )

        if val_metrics["acc"] > best_val_acc:
            best_val_acc = val_metrics["acc"]
            best_state = copy.deepcopy(model.state_dict())

    total_time = time.time() - total_start

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        use_amp=cfg.use_amp,
        desc="test",
    )

    if torch.cuda.is_available():
        peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
    else:
        peak_memory_mb = 0.0

    summary = {
        "attention": attention_type,
        "params": count_params(model),
        "best_val_acc": best_val_acc,
        "test_loss": test_metrics["loss"],
        "test_acc": test_metrics["acc"],
        "total_train_time_sec": total_time,
        "avg_epoch_train_time_sec": np.mean([h["train_time"] for h in history]),
        "peak_memory_mb": peak_memory_mb,
    }

    return model, pd.DataFrame(history), summary


Проверим сперва модели на 30 эпохах.

In [80]:
mhla_model, mhla_history, mhla_summary = run_experiment(
    attention_type="mhla",
    cfg=cfg,
)

mhla_summary



===== Experiment: mhla =====
Params: 751218

Epoch 1/30


/tmp/ipykernel_7000/1961669753.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_7000/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=3.3945 train_acc=0.2539 | val_loss=3.5512 val_acc=0.2049 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'mhla',
 'params': 751218,
 'best_val_acc': 0.2088235294117647,
 'test_loss': 3.691866265562302,
 'test_acc': 0.16815742397137745,
 'total_train_time_sec': 477.25461506843567,
 'avg_epoch_train_time_sec': np.float64(10.423238571484884),
 'peak_memory_mb': 2610.66650390625}

In [81]:
standard_model, standard_history, standard_summary = run_experiment(
    attention_type="standard",
    cfg=cfg,
)

standard_summary



===== Experiment: standard =====
Params: 745830

Epoch 1/30


/tmp/ipykernel_7000/1961669753.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_7000/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=3.2346 train_acc=0.2961 | val_loss=3.3948 val_acc=0.2657 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'standard',
 'params': 745830,
 'best_val_acc': 0.2696078431372549,
 'test_loss': 3.5189838992190334,
 'test_acc': 0.21076597820783868,
 'total_train_time_sec': 731.0605065822601,
 'avg_epoch_train_time_sec': np.float64(16.770416458447773),
 'peak_memory_mb': 10143.72998046875}

Как мы видим, оригинальное внимание при таком числе эпох показывает лучший результат (0.21 test acc против 0.16 на MHLS). При этом MHLS выигрывает по времени (10с на эпоху против 16 у классического внимания) и очень сильно выигрывает по использованию памяти (2610мб против 10143мб у классики).

Как мне кажется, пока эпох слишком мало для сравнения, модели могут еще обучаться. Поставим 60 эпох и посмотрим, что будет.

In [93]:
mhla_model, mhla_history, mhla_summary = run_experiment(
    attention_type="mhla",
    cfg=cfg,
)

mhla_summary



===== Experiment: mhla =====
Params: 751218

Epoch 1/60


/tmp/ipykernel_7000/1961669753.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_7000/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=2.7091 train_acc=0.4216 | val_loss=3.1757 val_acc=0.2951 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'mhla',
 'params': 751218,
 'best_val_acc': 0.3058823529411765,
 'test_loss': 3.3475339109867757,
 'test_acc': 0.2538624166531143,
 'total_train_time_sec': 962.9993426799774,
 'avg_epoch_train_time_sec': np.float64(10.44358323017756),
 'peak_memory_mb': 2610.66650390625}

In [94]:
standard_model, standard_history, standard_summary = run_experiment(
    attention_type="standard",
    cfg=cfg,
)

standard_summary



===== Experiment: standard =====
Params: 745830

Epoch 1/60


/tmp/ipykernel_7000/1961669753.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_7000/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=2.4769 train_acc=0.5098 | val_loss=2.9926 val_acc=0.3490 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'standard',
 'params': 745830,
 'best_val_acc': 0.35294117647058826,
 'test_loss': 3.141749489383012,
 'test_acc': 0.31419743047650023,
 'total_train_time_sec': 1477.7970967292786,
 'avg_epoch_train_time_sec': np.float64(16.96459967692693),
 'peak_memory_mb': 10109.04541015625}

Как мы видим, результаты в плане сравнения примерно такие же. Видно, что обе модели из-за малого числа параметров не могут хорошо обучиться. При этом модель на классическом внимании опять показала лучшую метрику в целом, но сильно проигрывает по времени на эпоху и использованию памяти.
Можно сделать вывод, что внимание MHLS выгодно использовать при малом числе ресурсов (особенно видеопамяти), при этом немного проигрывая в качестве.
Модель с MHLS вниманием можно сделать сильно больше по параметрам, чем модель с классическим вниманием, при этом получить одинаковое использование ресурсов моделями, тем самым возможно получить сопоставимое или лучшее качество.


### ДЗ7

Теперь скачаем репозиторий с SequenceLab вниманием. Эксперименты будем проводить с тем же корпусом, с тем же конфигом, чтобы сравнение было честным.

In [7]:
!git clone https://github.com/LJC-FVNR/SequenceLab.git
%cd SequenceLab
!pip install -e .
%cd ..


Cloning into 'SequenceLab'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 34 (delta 11), reused 25 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 30.69 KiB | 10.23 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/SequenceLab
Obtaining file:///content/SequenceLab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sequencelab (pyproject.toml) ... done
  Created wheel for sequencelab: filename=sequencelab-0.1.0-0.editable-py3-none-any.whl size=5993 sha256=732b6555ce35bb9757e131ce7c4fd56c80eb8af916f3225f1c0b28c52f5eb5d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-4d0bjx9i/wheels/16/52/55/829630f3803e253dadbec9bc0614b68ab2d7c167cd7e3738ee
Successfully built sequencelab
/cont

In [10]:
import sys
print(sys.executable)


/usr/bin/python3


In [11]:
%pip install -e /content/SequenceLab


Obtaining file:///content/SequenceLab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sequencelab (pyproject.toml) ... done
  Created wheel for sequencelab: filename=sequencelab-0.1.0-0.editable-py3-none-any.whl size=5993 sha256=6e6bd1b0e61787a87c7cda352b0077aec45d714f727cea9a914e8464e1c85d91
  Stored in directory: /tmp/pip-ephem-wheel-cache-484yd6c0/wheels/16/52/55/829630f3803e253dadbec9bc0614b68ab2d7c167cd7e3738ee
Successfully built sequencelab
  Attempting uninstall: sequencelab
    Found existing installation: sequencelab 0.1.0
    Uninstalling sequencelab-0.1.0:
      Successfully uninstalled sequencelab-0.1.0


In [14]:
import sys
import os
import importlib

seq_src = "/content/SequenceLab/src"

if seq_src not in sys.path:
    sys.path.insert(0, seq_src)

importlib.invalidate_caches()

from sequencelab.build import build_attention
from sequencelab.config import WaveConfig, ZeroSConfig, FEMConfig

print("SequenceLab imported successfully")


SequenceLab imported successfully


In [33]:
seq_fem_model, seq_fem_history, seq_fem_summary = run_experiment(
    attention_type="sequencelab_fem_softmax",
    cfg=cfg,
)

seq_fem_summary



===== Experiment: sequencelab_fem_softmax =====
Params: 696678

Epoch 1/60


/tmp/ipykernel_3864/3352663501.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_3864/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=2.4325 train_acc=0.5127 | val_loss=3.0143 val_acc=0.3549 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'sequencelab_fem_softmax',
 'params': 696678,
 'best_val_acc': 0.35784313725490197,
 'test_loss': 3.1643724975827885,
 'test_acc': 0.3081801919011221,
 'total_train_time_sec': 1274.1675953865051,
 'avg_epoch_train_time_sec': np.float64(14.50312800804774),
 'peak_memory_mb': 2578.87158203125}

Как мы видим, модель с вниманием SequenceLab показала примерно такую же точность на тесте, при этом среднее время на эпоху уменьшилось (16с -> 14с), использование памяти так же уменьшилось (10109мб -> 2578мб). По сравнению с вниманием из MHLS точность модели сильно выросла (0.25 acc -> 0.30 acc), при этом время на эпоху у MHLS тоже меньше (10с против 14с), потребление памяти примерно такое же. Можно сделать вывод, что модели с вниманием из SequenceLab сильно уменьшает потребление памяти, уменьшает время на эпоху, при этом почти не ухудшая точность ( в отличие от MHLS).

### ДЗ 8

Делаем все то же самое, что и для прошлых вариантов внимания

In [34]:
# ELSA attention
!git clone https://github.com/ming053l/ELSA.git
!pip install -e ./ELSA


Cloning into 'ELSA'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 114 (delta 29), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 274.78 KiB | 6.24 MiB/s, done.
Resolving deltas: 100% (29/29), done.
Obtaining file:///content/ELSA
  Preparing metadata (setup.py) ... done
  Running setup.py develop for elsa-attention


In [36]:
!pip install timm

import sys
import importlib

elsa_root = "/content/ELSA/code"

if elsa_root not in sys.path:
    sys.path.insert(0, elsa_root)

# чистим возможные неправильные импорты
for name in list(sys.modules.keys()):
    if name == "elsa" or name.startswith("stable"):
        del sys.modules[name]

importlib.invalidate_caches()

from stable.elsa import ElsaAttention

print("ELSA imported successfully")


ELSA imported successfully


/content/ELSA/code/stable/elsa_swin.py:2146: UserWarning: Overwriting elsa_tiny_window16_256 in registry with stable.elsa_swin.elsa_tiny_window16_256. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/ELSA/code/stable/elsa_swin.py:2172: UserWarning: Overwriting elsa_tiny_window8_256 in registry with stable.elsa_swin.elsa_tiny_window8_256. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/ELSA/code/stable/elsa_swin.py:2194: UserWarning: Overwriting elsa_small_window16_256 in registry with stable.elsa_swin.elsa_small_window16_256. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/ELSA/code/stable/elsa_swin.py:2218: UserWarning: Overwriting elsa_small_window8_256 in registry with stable.elsa_swin.elsa_small_window8_256. Thi

In [44]:
elsa_model = elsa_model.to(device)

images, targets = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    logits = elsa_model(images)

print(logits.shape)


/content/ELSA/code/stable/elsa.py:1510: RuntimeWarning: backend 'triton' mismatches runtime mode; using 'triton_fp32'.
  primary_backend = self._primary_backend_for_prepack(x)


torch.Size([64, 102])


/content/ELSA/code/stable/elsa.py:1510: RuntimeWarning: backend 'triton' mismatches runtime mode; using 'triton_fp32'.
  primary_backend = self._primary_backend_for_prepack(x)


In [45]:
elsa_model, elsa_history, elsa_summary = run_experiment(
    attention_type="elsa_auto",
    cfg=cfg,
)

elsa_summary



===== Experiment: elsa_auto =====
Params: 744678

Epoch 1/60


/tmp/ipykernel_3864/1060943201.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_3864/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):
/content/ELSA/code/stable/elsa.py:1510: RuntimeWarning: backend 'triton' mismatches runtime mode; using 'triton_fp32_train'.
  primary_backend = self._primary_backend_for_prepack(x)
/content/ELSA/code/stable/elsa.py:1510: RuntimeWarning: backend 'triton' mismatches runtime mode; using 'triton_fp32_train'.
  primary_backend = self._primary_backend_for_prepack(x)
/tmp/ipykernel_3864/1366650589.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(use_amp and device.type == "cuda")):


val:   0%|          | 0/16 [00:00<?, ?it/s]

...

train:   0%|          | 0/16 [00:00<?, ?it/s]

val:   0%|          | 0/16 [00:00<?, ?it/s]

train_loss=2.4739 train_acc=0.5275 | val_loss=3.0049 val_acc=0.3598 | lr=0.00e+00


test:   0%|          | 0/97 [00:00<?, ?it/s]

{'attention': 'elsa_auto',
 'params': 744678,
 'best_val_acc': 0.3637254901960784,
 'test_loss': 3.1730752886941054,
 'test_acc': 0.3016750691169296,
 'total_train_time_sec': 1240.968228340149,
 'avg_epoch_train_time_sec': np.float64(13.749565776189169),
 'peak_memory_mb': 2059.6767578125}

Модель с вниманием из ELSA показало схожие результаты с моделью с SequenceLab вниманием. Очень похожие результаты по accuracy (в районе 0.3), и времени на эпоху (около 14 секуннд), при этом использование памяти оказалось меньшим (2059мб против 2578мб).

### Итого

| Attention type | Params | Best val acc | Test loss | Test acc | Total train time, sec | Avg epoch train time, sec | Peak memory, MB |
|---|---:|---:|---:|---:|---:|---:|---:|
| MHLA | 751,218 | 0.3059 | 3.3475 | 0.2539 | 963.00 | 10.44 | 2610.67 |
| Standard | 745,830 | 0.3529 | 3.1417 | 0.3142 | 1477.80 | 16.96 | 10109.05 |
| SequenceLab FEM Softmax | 696,678 | 0.3578 | 3.1644 | 0.3082 | 1274.17 | 14.50 | 2578.87 |
| ELSA Auto | 744,678 | 0.3637 | 3.1731 | 0.3017 | 1240.97 | 13.75 | 2059.68 |
|


Лучшее качество на тестовой выборке показала модель с классическим вниманием:

- `test_acc = 0.3142`;
- `best_val_acc = 0.3529`.

Однако классическое внимание оказалось самым дорогим по ресурсам:

- среднее время эпохи — `16.96` секунд;
- пиковое потребление памяти — `10109.05` MB.

Самым быстрым вариантом оказалось MHLA внимание:

- среднее время эпохи — `10.44` секунд;
- общее время обучения — `963.00` секунд.

Но при этом MHLA показало худшее качество:

- `test_acc = 0.2539`;
- `best_val_acc = 0.3059`.

То есть MHLA действительно даёт выигрыш по скорости и памяти, но в данном эксперименте заметно проигрывает по точности классификации.


SequenceLab FEM Softmax показало хороший компромисс между качеством и ресурсами. Его точность почти совпала с классическим attention:

- `test_acc = 0.3082` против `0.3142` у standard attention.

При этом потребление памяти оказалось почти в 4 раза меньше:

- `2578.87` MB против `10109.05` MB.

Среднее время эпохи также меньше, чем у классического attention:

- `14.50` секунд против `16.96` секунд.

ELSA Auto показало лучший результат на validation среди всех моделей:

- `best_val_acc = 0.3637`.

На тесте ELSA немного уступила классическому attention и SequenceLab FEM:

- `test_acc = 0.3017`.

Зато ELSA оказалась самой экономной по памяти:

- `2059.68` MB.

Также ELSA обучалась быстрее стандартного внимания:

- `13.75` секунд на эпоху против `16.96` секунд.

Таким образом, ELSA дала самый сильный выигрыш по памяти при сохранении приемлемого качества. По сравнению с классическим attention она уменьшила пиковое потребление памяти примерно с `10109` MB до `2060` MB, то есть почти в 5 раз, а время эпохи уменьшилось примерно на 19%. При этом падение test accuracy составило около `0.0125`.